# Fly Kev on the Low Gate benchmark

Runs one Kev model (the open-weight, Jev-style decision family, [github.com/jaredpalmer/kev](https://github.com/jaredpalmer/kev)) on a Colab GPU and flies it over the benchmark courses 5001–5020, exactly as `pnpm bench` does locally. Kev's server and the benchmark run on the same Colab machine, so nothing is exposed to the internet.

**Before you start**
- Runtime → Change runtime type → **G4 GPU** (NVIDIA RTX PRO 6000, 96 GB) for every Kev lane, with High-RAM on if offered. Kev needs bf16, which the T4 lacks. Kev's README lists an L4 for 0.8B only, an L40S or H100 for 4B and 9B, and an 80 GB GPU for 27B (about 54 GB of bf16 weights); the G4 holds every Kev size. One GPU type for all lanes keeps their answer times comparable (Laya's check runs on it too).
- Only for a private fork: Colab's key icon (Secrets) → add `GITHUB_TOKEN`, a fine-grained token with **read-only Contents** access to that repository, and set `REPO` in step 3. The notebook never prints it.
- Set `LANE` in the next cell, then Runtime → Run all. For the next lane, change `LANE` and run all again in the same runtime: step 5 stops the previous Kev first.

**What you get:** `<LANE>-bench.zip` downloads at the end. Unzip it into the repo root on your computer (it holds `bench/<LANE>/*.json`), then run `pnpm bench --summary`. The benchmark is resumable: if Colab disconnects, run all again and it carries on from the courses already in `bench/` (download the zip before the runtime is recycled).

Answer times measure this Colab GPU, not a hosted service; the results page labels them that way.

In [ ]:
LANE = "kev-4b"   # "kev-0.8b", "kev-4b", "kev-9b" or "kev-27b"
RUN = {"kev-0.8b": "jaredpalmer/kev-0.8b@v1.0", "kev-4b": "jaredpalmer/kev-4b@v1.0", "kev-9b": "jaredpalmer/kev-9b@v1.0", "kev-27b": "jaredpalmer/kev-27b@v1.0"}[LANE]
PORT = 8009       # the Kev lanes' upstream in packages/decisions/src/catalog.ts
NODE = "v24.21.0" # same Node as the machine that flew the other lanes

In [ ]:
# 1. Check the GPU: Kev serves in bf16.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → G4 GPU"
assert torch.cuda.is_bf16_supported(), "This GPU has no bf16 (a T4?): pick a G4"

In [ ]:
# 2. Node and pnpm, pinned to the versions the local lanes were flown with.
!curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C /opt
import os
os.environ["PATH"] = f"/opt/node-{NODE}-linux-x64/bin:" + os.environ["PATH"]
os.environ["COREPACK_ENABLE_DOWNLOAD_PROMPT"] = "0"
!corepack enable && node -v && pnpm -v

In [ ]:
# 3. The Low Gate repo. A GITHUB_TOKEN in Colab Secrets is used only if present (for a private fork); it stays out of the output and out of .git/config.
from google.colab import userdata
import subprocess
REPO = "github.com/ErenTerakye/lowgate.git"
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
if not os.path.exists("/content/lowgate"):
    url = f"https://x-access-token:{token}@{REPO}" if token else f"https://{REPO}"
    r = subprocess.run(["git", "clone", "-q", url, "/content/lowgate"], capture_output=True, text=True)
    assert r.returncode == 0, "Clone failed: check the repository URL (and GITHUB_TOKEN, if the repository is private)"
    subprocess.run(["git", "-C", "/content/lowgate", "remote", "set-url", "origin", f"https://{REPO}"], check=True)
del token
%cd /content/lowgate
!git log --oneline -1 && pnpm install --frozen-lockfile --silent && pnpm -s test 2>&1 | tail -3

In [ ]:
# 4. Kev's server, from source (the README warns that `pip install kev` is an unrelated package).
!pip install -q uv
!test -d /content/kev || git clone -q https://github.com/jaredpalmer/kev.git /content/kev
%cd /content/kev
!git log --oneline -1
!uv sync --extra serve 2>&1 | tail -2

In [ ]:
# 5. Start Kev in the background on this machine only; the first start downloads the weights from Hugging Face.
#    A Kev left over from an earlier LANE in this runtime is stopped first: it would keep the port, the new server could
#    not start, and the old model would answer every question (this happened on 2026-10-31).
import time, json, urllib.request
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=60)
subprocess.run(["pkill", "-f", "kev.serve"])
time.sleep(3)
server = subprocess.Popen(
    ["uv", "run", "--extra", "serve", "python", "-m", "kev.serve", "--run", RUN, "--port", str(PORT), "--device", "cuda"],
    cwd="/content/kev", stdout=open("/content/kev.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(180):
    assert server.poll() is None, "Kev stopped: see the log below\n" + open("/content/kev.log").read()[-3000:]
    try:
        models = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=5).read())["models"]
        break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError("Kev did not start in 30 minutes:\n" + open("/content/kev.log").read()[-3000:])
# The pin check: the checkpoint this server loaded. Answers can't show it, because Kev echoes the requested model name.
served_run = models[0].get("run")
print("server runs:", served_run, "| base:", models[0].get("base"), "| dtype:", models[0].get("dtype"))
assert served_run == RUN, f"The server on port {PORT} runs {served_run}, not {RUN}: restart the runtime and run all again"
!nvidia-smi --query-gpu=memory.used --format=csv

In [ ]:
# 6. One raw request, to see an answer and its latency. Kev echoes the model name the request asks for, so this name is
#    not a pin check; step 5's check of the loaded checkpoint is.
req = {"model": RUN, "state": "A lander is 40 m above the pad, falling at 5 m/s.",
       "questions": {"burn": {"type": "noul", "instructions": "Should it fire the engine now?"}}}
r = urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/systemone", json.dumps(req).encode(),
                                                  {"Content-Type": "application/json"}), timeout=60)
body = json.loads(r.read())
print("served model:", body.get("model"), "| pin:", RUN, "| latency_ms:", body.get("latency_ms"))

In [ ]:
# 7. Probe: level 1 of course 42 (a tuning course, not a benchmark one), to see Kev fly before the long run.
%cd /content/lowgate
!pnpm -s probe --lane {LANE} --levels 1 --seed 42

In [ ]:
# 8. The benchmark: courses 5001–5020. Resumable; each course is saved only after its replay check passes.
!pnpm -s bench --lanes {LANE}

In [ ]:
# 9. Download the recordings and the server log. Unzip in the repo root on your computer (the log lands there and git ignores it), then `pnpm bench --summary`.
%cd /content/lowgate
!cp /content/kev.log {LANE}-server.log && zip -qr /content/{LANE}-bench.zip bench/{LANE} {LANE}-server.log
from google.colab import files
files.download(f"/content/{LANE}-bench.zip")